In [1]:
from ftplib import FTP_TLS
import os

# ===== 共通設定 =====
ftp_host = "sv16194.xserver.jp"
ftp_user = "sedoinfinity"
ftp_pass = "d958b40323"

today = "20250716"

# ===== ローカルパス =====
if os.name == 'nt':
    user_base = os.path.join(os.environ["USERPROFILE"], "myenv", "domz")
else:
    user_base = os.path.join(os.path.expanduser("~"), "myenv", "domz")

local_html_dir = user_base
local_webp_dir = os.path.join(user_base, "pic", today, "webp")

# ===== リモートパス =====
remote_html_dir = "sedoinfinity.xsrv.jp/public_html/domz/"
remote_webp_dir = f"sedoinfinity.xsrv.jp/public_html/domz/pic/{today}/webp/"

# ✅ FTPS接続
ftps = FTP_TLS(ftp_host)
ftps.login(ftp_user, ftp_pass)
ftps.prot_p()
print("[INFO] ✅ FTPS接続完了")


# ✅ HTMLアップロード（index系・today含む・machines配下）
def upload_html_smart(local_dir, remote_dir, today):
    print(f"[INFO] ✅ HTML選別アップロード開始 {local_dir} → {remote_dir}")

    exclude_dirs = ["pic", "db", "templates", "scripts", ".ipynb_checkpoints"]

    ftps.cwd("/")
    for part in remote_dir.strip("/").split("/"):
        ftps.cwd(part)

    for root, dirs, files in os.walk(local_dir):
        if any(skip in root for skip in exclude_dirs):
            continue

        rel_path = os.path.relpath(root, local_dir)

        # サブフォルダ作成＆移動
        if rel_path != ".":
            for part in rel_path.split(os.sep):
                try:
                    ftps.mkd(part)
                except Exception:
                    pass
                ftps.cwd(part)

        for file in files:
            if not file.endswith(".html"):
                continue

            file_path = os.path.join(root, file)

            is_index = file.startswith("index")
            is_today = today in file
            is_machine_html = "machines" in os.path.normpath(root)

            # ✅ today含むものは必ずアップロード
            if not (is_index or is_today or is_machine_html):
                continue

            with open(file_path, 'rb') as f:
                ftps.storbinary(f'STOR ' + file, f)
            print(f"[INFO] ✅ アップロード: {os.path.join(rel_path, file)}")

        if rel_path != ".":
            for _ in rel_path.split(os.sep):
                ftps.cwd("..")

    ftps.cwd("/")
    print("[INFO] ✅ HTML選別アップロード完了\n")


# ✅ webpアップロード
def upload_webp(local_dir, remote_dir):
    print(f"[INFO] ✅ webpフォルダアップロード開始 {local_dir} → {remote_dir}")
    ftps.cwd("/")
    for part in remote_dir.strip("/").split("/"):
        try:
            ftps.cwd(part)
        except Exception:
            ftps.mkd(part)
            ftps.cwd(part)

    for filename in os.listdir(local_dir):
        path = os.path.join(local_dir, filename)
        if os.path.isfile(path):
            with open(path, 'rb') as f:
                ftps.storbinary(f'STOR ' + filename, f)
            print(f"[INFO] ✅ アップロード: {filename}")

    ftps.cwd("/")
    print("[INFO] ✅ webpアップロード完了\n")


# ✅ 実行
upload_html_smart(local_html_dir, remote_html_dir, today)
upload_webp(local_webp_dir, remote_webp_dir)

ftps.quit()
print("[INFO] ✅ ✅ アップロード＆切断 完了")


[INFO] ✅ FTPS接続完了
[INFO] ✅ HTML選別アップロード開始 C:\Users\stray\myenv\domz → sedoinfinity.xsrv.jp/public_html/domz/
[INFO] ✅ アップロード: .\index.html
[INFO] ✅ アップロード: machines\machine_1000.html
[INFO] ✅ アップロード: machines\machine_1001.html
[INFO] ✅ アップロード: machines\machine_1002.html
[INFO] ✅ アップロード: machines\machine_1003.html
[INFO] ✅ アップロード: machines\machine_1005.html
[INFO] ✅ アップロード: machines\machine_1006.html
[INFO] ✅ アップロード: machines\machine_1007.html
[INFO] ✅ アップロード: machines\machine_1008.html
[INFO] ✅ アップロード: machines\machine_1010.html
[INFO] ✅ アップロード: machines\machine_1011.html
[INFO] ✅ アップロード: machines\machine_1012.html
[INFO] ✅ アップロード: machines\machine_1013.html
[INFO] ✅ アップロード: machines\machine_1015.html
[INFO] ✅ アップロード: machines\machine_1016.html
[INFO] ✅ アップロード: machines\machine_1017.html
[INFO] ✅ アップロード: machines\machine_1018.html
[INFO] ✅ アップロード: machines\machine_1020.html
[INFO] ✅ アップロード: machines\machine_1021.html
[INFO] ✅ アップロード: machines\machine_1022.html
[INFO] ✅ アップロード: machines

EOFError: 